<a href="https://colab.research.google.com/github/SakshamStha47/Bus_Stop_Accessibility_Analysis-Old_West_Side/blob/main/notebooks/03_microtransit_sensitivity_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. SETUP

## 1.1 MOUNT DRIVE & CONFIGURE R5PY MEMORY

Mount drive to access saved files

In [16]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.argv.extend(["--max-memory", "10G"])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Install latest version of JAVA

In [ ]:
import os

# 1. Force non-interactive installation and remove the silent flags to monitor progress
!DEBIAN_FRONTEND=noninteractive apt-get update
!DEBIAN_FRONTEND=noninteractive apt-get install -y openjdk-21-jdk

# 2. Set update-alternatives
!update-alternatives --install /usr/bin/java java /usr/lib/jvm/java-21-openjdk-amd64/bin/java 100
!update-alternatives --install /usr/bin/javac javac /usr/lib/jvm/java-21-openjdk-amd64/bin/javac 100
!update-alternatives --set java /usr/lib/jvm/java-21-openjdk-amd64/bin/java
!update-alternatives --set javac /usr/lib/jvm/java-21-openjdk-amd64/bin/javac

# 3. Configure Environment Variables
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

# 4. Verify
!java -version

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [84.1 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,155 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,314 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Pac

In [ ]:
!pip install r5py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.8/63.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.1/290.1 kB 11.8 MB/s eta 0:00:00


## 1.2 IMPORTS & ENVIRONMENT CONFIG

In [19]:
import shutil
import os

src = "/content/drive/MyDrive/Colab Notebooks/Project Dream/r5py_cache_backup"
dst = os.path.expanduser("~/.cache/r5py")

# Ensure the destination directory is clean before copying
if os.path.exists(dst):
    shutil.rmtree(dst)
os.makedirs(dst, exist_ok=True)
shutil.copytree(src, dst, dirs_exist_ok=True, symlinks=True)
print("Cache restored from Drive")
print(os.listdir(dst))

Cache restored from Drive
['subway.zip', 'manhattan_bus.zip', 'brooklyn_bus.zip', 'manhattan_bus.zip.lock', 'subway.zip.lock', 'bronx_bus.zip.lock', 'f7f5e1b757f61243ed56ab15c2d4f1bf05fb6304c54e41f5403a1121f72b3e83.transport_network', 'bronx_bus.zip', 'brooklyn_bus.zip.lock', 'queens_bus.zip', 'f7f5e1b757f61243ed56ab15c2d4f1bf05fb6304c54e41f5403a1121f72b3e83.warnings', 'f7f5e1b757f61243ed56ab15c2d4f1bf05fb6304c54e41f5403a1121f72b3e83.mapdb.p', 'queens_bus.zip.lock', 'r5-v7.5.1-r5py-all.jar', 'nyc_routable.osm.pbf', 'f7f5e1b757f61243ed56ab15c2d4f1bf05fb6304c54e41f5403a1121f72b3e83.mapdb', 'nyc_routable.osm.pbf.lock']


## 1.3 RESTORE/BUILD TRANSPORT NETWORK

In [ ]:
import r5py

transport_network = r5py.TransportNetwork(
    osm_pbf="/root/.cache/r5py/nyc_routable.osm.pbf",
    gtfs=[
        "/root/.cache/r5py/bronx_bus.zip",
        "/root/.cache/r5py/brooklyn_bus.zip",
        "/root/.cache/r5py/queens_bus.zip",
        "/root/.cache/r5py/manhattan_bus.zip",
        "/root/.cache/r5py/subway.zip",
    ],
)
print("Transport network ready")

# 2. DATA LOADING

## 2.1 Load Tract Geometries, Equity Scores & Quadrant Labels

In [ ]:
import geopandas as gpd

gdf_clean = gpd.read_parquet("/content/drive/MyDrive/Colab Notebooks/Project Dream/Data/gdf_clean_full_variables.parquet")

print(gdf_clean.columns.to_list())

['STATEFP', 'COUNTYFP', 'TRACTCE', 'GEOID', 'NAME', 'NAMELSAD', 'MTFCC', 'FUNCSTAT', 'ALAND', 'AWATER', 'INTPTLAT', 'INTPTLON', 'geometry', 'PC1_Economic_Need', 'PC2_Mobility_Need', 'PC3_Employment_Gap', 'Transit_Equity_Score', 'reachable_jobs', 'quadrant', 'dist_subway_m', 'dist_bus_m', 'subway_1km', 'bus_stops_1km', 'total_population', 'pop_density', 'dist_cbd_m']


## 2.2 Load Job Data

In [ ]:
nyc_tract_jobs = gpd.read_parquet("/content/drive/MyDrive/Colab Notebooks/Project Dream/Data/nyc_tract_jobs.parquet")

print(nyc_tract_jobs.columns.to_list())

['STATEFP', 'COUNTYFP', 'TRACTCE', 'GEOID', 'GEOIDFQ', 'NAME', 'NAMELSAD', 'MTFCC', 'FUNCSTAT', 'ALAND', 'AWATER', 'INTPTLAT', 'INTPTLON', 'geometry', 'tract_id', 'total_jobs']


## 2.3 Load AM-Peak Baseline Accessibility

In [ ]:
import pandas as pd

# Load AM-peak matrix and apply the known overshoot filter
job_accessibility = pd.read_parquet(
    f"/content/drive/MyDrive/Colab Notebooks/Project Dream/Data/travel_time_matrix_am_peak_weekday_20260714_0800.parquet"
)
job_accessibility_filtered = job_accessibility[job_accessibility["travel_time"] <= 45].copy()

# Merge job counts onto destinations
job_accessibility_merged = job_accessibility_filtered.merge(
    nyc_tract_jobs[["tract_id", "total_jobs"]].rename(columns={"tract_id": "to_id"}),
    on="to_id",
    how="left"
)
job_accessibility_merged["total_jobs"] = job_accessibility_merged["total_jobs"].fillna(0)

# Sum reachable jobs per origin tract
reachable_jobs = (
    job_accessibility_merged.groupby("from_id")["total_jobs"]
    .sum()
    .reset_index()
    .rename(columns={"from_id": "GEOID", "total_jobs": "reachable_jobs"})
)

print(reachable_jobs.shape)

(805, 2)


In [ ]:
gdf_clean = gdf_clean.merge(reachable_jobs, on="GEOID", how="left")
print(gdf_clean["reachable_jobs"].isna().sum(), "tracts missing job accessibility")

gdf_clean = gdf_clean.dropna(subset=["Transit_Equity_Score", "reachable_jobs"]).copy()
print(f"Tracts after dropping NaN: {len(gdf_clean)}")

0 tracts missing job accessibility
Tracts after dropping NaN: 774


In [ ]:
need_median = gdf_clean["Transit_Equity_Score"].median()
access_median = gdf_clean["reachable_jobs"].median()

def assign_quadrant(row):
    high_need = row["Transit_Equity_Score"] > need_median
    high_access = row["reachable_jobs"] > access_median
    if high_need and not high_access:
        return "High Need, Low Access"
    elif high_need and high_access:
        return "High Need, High Access"
    elif not high_need and not high_access:
        return "Low Need, Low Access"
    else:
        return "Low Need, High Access"

gdf_clean["quadrant"] = gdf_clean.apply(assign_quadrant, axis=1)
print(gdf_clean["quadrant"].value_counts())

quadrant
Low Need, High Access     222
High Need, Low Access     222
Low Need, Low Access      165
High Need, High Access    165
Name: count, dtype: int64


## 2.4 Load & Clean Subway Stop Data (deduplicated, Brooklyn-clipped)

Raw GTFS subway stops include duplicate platform-level entries per physical
station (e.g. parent stop + N/S directional stops). This step deduplicates to
one row per real station and clips to Brooklyn, giving 167 stations.

In [ ]:
import zipfile, io

with zipfile.ZipFile("/root/.cache/r5py/subway.zip") as z:
    stops_raw = pd.read_csv(io.BytesIO(z.read("stops.txt")))

# Keep only parent stations (deduplicates platform-level N/S entries)
stops_dedup = stops_raw[stops_raw["parent_station"].isna()].copy()

stops_dedup_gdf = gpd.GeoDataFrame(
    stops_dedup,
    geometry=gpd.points_from_xy(stops_dedup["stop_lon"], stops_dedup["stop_lat"]),
    crs="EPSG:4326"
).to_crs(epsg=3857)

# Clip to Brooklyn
gdf_3857 = gdf_clean.to_crs(epsg=3857)
brooklyn_boundary = gdf_3857.union_all()
destinations_subway_clean = stops_dedup_gdf[stops_dedup_gdf.within(brooklyn_boundary)].copy()

destinations_subway_clean["id"] = destinations_subway_clean["stop_id"].astype(str)
destinations_subway_clean = destinations_subway_clean[["id", "geometry"]].to_crs("EPSG:4326")

print(f"Deduplicated, Brooklyn-clipped subway stations: {len(destinations_subway_clean)}")

OSError: [Errno 40] Too many levels of symbolic links: '/root/.cache/r5py/subway.zip'

## 2.5 Define Target Population (High Need, Low Access tracts)

In [ ]:
low_access_geoids = gdf_clean[gdf_clean["quadrant"] == "High Need, Low Access"]["GEOID"]
print(f"Target tracts (High Need, Low Access): {len(low_access_geoids)}")

# Tract centroids as origins (projected for accurate centroid calc, then back to WGS84)
gdf_proj = gdf_clean.to_crs(epsg=32618)
gdf_proj["centroid"] = gdf_proj.geometry.centroid

origins_centroids = gdf_proj[["GEOID", "centroid"]].rename(columns={"GEOID": "id", "centroid": "geometry"})
origins_centroids = gpd.GeoDataFrame(origins_centroids, geometry="geometry", crs=gdf_proj.crs).to_crs("EPSG:4326")

print(origins_centroids.shape)
print(origins_centroids.columns.to_list())

In [ ]:
destinations = nyc_tract_jobs.copy()
destinations["geometry"] = destinations.to_crs(epsg=32618).geometry.centroid.to_crs("EPSG:4326")
destinations = destinations[["tract_id", "geometry"]].rename(columns={"tract_id": "id"})
destinations = destinations.drop_duplicates(subset="id", keep="first").copy()

print(destinations.shape)
print(destinations.columns.to_list())

# 3. FIRST-MILE TRAVEL TIME COMPARISON BETWEEN HIGH-NEED-HIGH-ACCESS AND HIGH-NEED-LOW-ACCESS TRACTS

## 3.1. Create a quadrant subset of high need, high access and high need, low access

In [ ]:
comparison_tracts = gdf_clean[
    gdf_clean["quadrant"].isin(["High Need, Low Access", "High Need, High Access"])
].copy()

print(comparison_tracts["quadrant"].value_counts())

## 3.2. Define origin based on the subset

In [ ]:
comparison_tracts_proj = comparison_tracts.to_crs(epsg=32618)  # projected, in meters
comparison_tracts_proj["geometry"] = comparison_tracts_proj.geometry.centroid
origins_centroids_subset = comparison_tracts_proj[["GEOID", "quadrant", "geometry"]].rename(columns={"GEOID": "id"})
origins_centroids_subset = origins_centroids.to_crs("EPSG:4326")  # r5py expects WGS84

## 3.3. Calculate first-mile travel time matrix

In [ ]:
from datetime import datetime, timedelta

first_mile_matrix_bus = r5py.TravelTimeMatrix(
    transport_network,
    origins=origins_centroids_subset[["id", "geometry"]],  # r5py needs just id + geometry
    destinations=destinations_subway_clean,
    departure=datetime(2026, 7, 14, 8, 0, 0),
    departure_time_window=timedelta(hours=1),
    transport_modes=[r5py.TransportMode.BUS, r5py.TransportMode.WALK],      # Travel times are computed through a combination of walking and bus trips
)

# Isolate nearest subway stations (stations that can be reached in shortest time)
nearest_subway_time_bus = (
    first_mile_matrix_bus.groupby("from_id")["travel_time"]
    .min()
    .reset_index()
    .rename(columns={"from_id": "GEOID", "travel_time": "first_mile_time_min"})
)

# Merge quadrant label back in for comparison
nearest_subway_time_bus = nearest_subway_time_bus.merge(
    comparison_tracts[["GEOID", "quadrant"]], on="GEOID", how="left"
)

print(nearest_subway_time_bus.groupby("quadrant")["first_mile_time_min"].describe())

## 3.4. Visualize the data using box plots

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(8, 6))
sns.boxplot(data=nearest_subway_time_bus, x="quadrant", y="first_mile_time_min", ax=ax)
sns.stripplot(data=nearest_subway_time_bus, x="quadrant", y="first_mile_time_min", color="black", alpha=0.3, size=3, ax=ax)
ax.set_ylabel("First-mile walk time to nearest subway (minutes)")
ax.set_xlabel("")
ax.set_title("First-Mile Access Time: High Need, Low Access vs. High Need, High Access")
plt.tight_layout()
plt.show()

# 4. MICRO-TRANSIT CATCHMENT COMPUTATION

## 4.1 Tract Centroids -> All Subway Stations (Car Mode, Uncapped)

Computes car-mode travel time from each high-need-low-access tract's centroid to every Brooklyn subway station, with no distance/time cap applied.

In [ ]:
import numpy as np
import r5py
from datetime import datetime, timedelta

origin_subset = origins_centroids[origins_centroids["id"].isin(low_access_geoids)][["id", "geometry"]]
batches = np.array_split(origin_subset, 5)

all_results = []
for i, batch in enumerate(batches):
    print(f"Processing batch {i+1}/{len(batches)} ({len(batch)} tracts)...")
    batch_matrix = r5py.TravelTimeMatrix(
        transport_network,
        origins=batch,                                # Centroids of low access tracts
        destinations=destinations_subway_clean,       # All subway stations within Brooklyn
        departure=datetime(2026, 7, 14, 8, 0, 0),
        departure_time_window=timedelta(hours=1),
        transport_modes=[r5py.TransportMode.CAR],
    )
    all_results.append(batch_matrix)
    print(f"  Done — {len(batch_matrix)} rows")

microtransit_raw_all = pd.concat(all_results, ignore_index=True)
microtransit_raw_all = microtransit_raw_all.rename(
    columns={"from_id": "GEOID", "to_id": "station_id", "travel_time": "microtransit_time"}
)

print(f"\nFinal shape (uncapped): {microtransit_raw_all.shape}")
print(microtransit_raw_all["microtransit_time"].describe())

## 4.2 Save Raw Catchment Matrix

In [ ]:
data_dir ="/content/drive/MyDrive/Colab Notebooks/Project Dream/Data"

microtransit_raw_all.to_parquet(f"{data_dir}/microtransit_raw_uncapped_dedup.parquet")
print("Saved microtransit_raw_uncapped_dedup.parquet to Drive")

# 5. STATION-TO-JOB MATRIX

## 5.1 Identify Relevant Stations
Rather than computing transit routes from all 169 Brooklyn stations, we only
need routes from stations that actually fall within a plausible microtransit
catchment of at least one target tract. This keeps the next (expensive)
computation scoped to what's actually needed.

In [ ]:
# Use a generous upper-bound cutoff here (20 min) so this station set covers
MAX_CATCHMENT_FOR_STATION_SELECTION = 20

relevant_stations = microtransit_raw_all[
    microtransit_raw_all["microtransit_time"] <= MAX_CATCHMENT_FOR_STATION_SELECTION
]["station_id"].unique()

station_origins_filtered = destinations_subway_clean[
    destinations_subway_clean["id"].isin(relevant_stations)
]

print(f"Stations relevant to at least one target tract (≤{MAX_CATCHMENT_FOR_STATION_SELECTION} min): "
      f"{len(station_origins_filtered)} (of {len(destinations_subway_clean)} total)")

## 5.2 Compute Station -> Job Destination Travel Times (Transit + Walk)

In [ ]:
station_batches = np.array_split(station_origins_filtered[["id", "geometry"]], 3)

all_station_results = []
for i, batch in enumerate(station_batches):
    print(f"Processing station batch {i+1}/{len(station_batches)} ({len(batch)} stations)...")
    batch_result = r5py.TravelTimeMatrix(
        transport_network,
        origins=batch,                                # Subway stations that are reachable with 20 mins
        destinations=destinations,                    # All job locations within NYC
        departure=datetime(2026, 7, 14, 8, 0, 0),
        departure_time_window=timedelta(hours=1),
        transport_modes=[r5py.TransportMode.TRANSIT, r5py.TransportMode.WALK],
    )
    all_station_results.append(batch_result)
    print(f"  Done — {len(batch_result)} rows")

station_to_jobs_all = pd.concat(all_station_results, ignore_index=True)
station_to_jobs_all = station_to_jobs_all.rename(
    columns={"from_id": "station_id", "to_id": "job_tract_id", "travel_time": "transit_time"}
)

print(f"\nFinal shape: {station_to_jobs_all.shape}")

## 5.3 Save Matrix

In [ ]:
station_to_jobs_all.to_parquet(f"{data_dir}/station_to_jobs_all_dedup.parquet")
print("Saved station_to_jobs_all_dedup.parquet to Drive")

# 6. CORE ACCESSIBILITY MODEL

## 6.1 Apply Catchment Cutoff (10 min) & Top-N Station Restriction (N=3)

Baseline parameters, justified empirically in Sections 6–7:
- 10-minute catchment: shortest duration at which every target tract has
  access to at least 3 candidate stations
- Top-3 stations: stays below the High-Access plausibility benchmark while
  approximating realistic small-shuttle service design (2–4 fixed hub routes)

In [ ]:
CATCHMENT_MINUTES = 10
N_STATIONS = 3

microtransit_capped = microtransit_raw_all[                             # First-mile OD data with travel time within 10 mins
    microtransit_raw_all["microtransit_time"] <= CATCHMENT_MINUTES
].copy()

top_n_stations = (
    microtransit_capped.sort_values("microtransit_time")
    .groupby("GEOID")
    .head(N_STATIONS)
    .copy()
)

# Verify restriction applied correctly before proceeding
max_stations_check = top_n_stations.groupby("GEOID").size().max()
print(f"Max stations per tract (should be ≤{N_STATIONS}): {max_stations_check}")
assert max_stations_check <= N_STATIONS, "Top-N restriction not applied correctly!"
print(f"Restriction verified! — {top_n_stations['GEOID'].nunique()} tracts covered")

## 6.2 Compute Best Route per Tract–Destination Pair

For each tract, tests all of its top-N candidate stations against every job
destination and keeps the fastest combined route.

In [ ]:
import gc

results = []
for i, geoid in enumerate(low_access_geoids):
    tract_stations = top_n_stations[top_n_stations["GEOID"] == geoid]             # Select the current (ith) row in the loop
    if tract_stations.empty:                                                      # Skip if the row has access to zero stations
        continue

    tract_combined = tract_stations.merge(station_to_jobs_all, on="station_id")                               # Merge station->job destinations data for available tracts
    tract_combined["total_time"] = tract_combined["microtransit_time"] + tract_combined["transit_time"]       # Sum of first-mile travel time and actual transit time

    tract_combined = tract_combined.dropna(subset=["total_time"])
    if tract_combined.empty:
        continue

    best_per_destination = tract_combined.loc[tract_combined.groupby("job_tract_id")["total_time"].idxmin()]    # Group all rows by destination tract ID and keep only the row with minimum travel time
    results.append(best_per_destination)

    if (i + 1) % 50 == 0:
        print(f"Processed {i+1}/{len(low_access_geoids)} tracts")

best_route = pd.concat(results, ignore_index=True)
gc.collect()

# Re-verify the restriction held through the merge
final_check = best_route.groupby("GEOID")["station_id"].nunique().max()
print(f"\nFinal shape: {best_route.shape}")
print(f"Max distinct stations used per tract (should be ≤{N_STATIONS}): {final_check}")
assert final_check <= N_STATIONS, "Something broke the top-N restriction downstream!"

## 6.3 Apply 45-Minute Threshold & Compute Reachable Jobs

In [ ]:
best_route_filtered = best_route[best_route["total_time"] <= 45].copy()                           # Filter out routes that exceed the 45 minute threshold.

best_route_with_jobs = best_route_filtered.merge(                                                 # Add tract id and job count to the filtered routes data.
    nyc_tract_jobs[["tract_id", "total_jobs"]].rename(columns={"tract_id": "job_tract_id"}),
    on="job_tract_id",
    how="left"
)
best_route_with_jobs["total_jobs"] = best_route_with_jobs["total_jobs"].fillna(0)

reachable_jobs_microtransit = (                                                                    # Sum the total jobs in every unique tract inside the filtered route data.
    best_route_with_jobs.groupby("GEOID")["total_jobs"]
    .sum()
    .reset_index()
    .rename(columns={"total_jobs": "reachable_jobs_microtransit"})
)

print(reachable_jobs_microtransit.describe())

## 6.4 Sanity Check vs. High-Access Benchmark

In [ ]:
high_access_baseline = gdf_clean[gdf_clean["quadrant"] == "High Need, High Access"]["reachable_jobs"]
low_access_baseline = gdf_clean[gdf_clean["quadrant"] == "High Need, Low Access"]["reachable_jobs"]

print(f"High Access baseline — median: {high_access_baseline.median():,.0f}, max: {high_access_baseline.max():,.0f}")
print(f"Low Access baseline (pre-intervention) — median: {low_access_baseline.median():,.0f}")
print(f"Low Access POST-microtransit — median: {reachable_jobs_microtransit['reachable_jobs_microtransit'].median():,.0f}, "
      f"max: {reachable_jobs_microtransit['reachable_jobs_microtransit'].max():,.0f}")

assert reachable_jobs_microtransit["reachable_jobs_microtransit"].median() < high_access_baseline.median(), \
    "Median post-intervention accessibility exceeds High-Access benchmark — check assumptions!"
print("\n Result stays below High-Access benchmark \n Verdict: Plausible!")

In [ ]:
reachable_jobs_microtransit.to_parquet(f"{data_dir}/reachable_jobs_microtransit_baseline.parquet")
best_route.to_parquet(f"{data_dir}/best_route_baseline.parquet")
print("Saved baseline microtransit results to Drive")

# 7. SENSITIVITY ANALYSIS: CATCHMENT DURATION

## 7.1 Sweep Catchment Duration (3–20 min), Fixed N=3

Tests whether the 10-minute catchment choice is justified, by holding the
top-3 station restriction fixed and varying only how far the catchment search
extends.

In [ ]:
x = range(1, 20)
x

In [ ]:
catchment_values = list(range(1, 21))
catchment_sweep_results = []

for cutoff in catchment_values:
    capped = microtransit_raw_all[microtransit_raw_all["microtransit_time"] <= cutoff].copy()

    # Track how many tracts fall short of having 3 stations available at this cutoff
    counts = capped.groupby("GEOID").size()
    tracts_below_n = (counts < N_STATIONS).sum()

    top_n_at_cutoff = capped.sort_values("microtransit_time").groupby("GEOID").head(N_STATIONS)

    results = []
    for geoid in low_access_geoids:
        tract_stations = top_n_at_cutoff[top_n_at_cutoff["GEOID"] == geoid]
        if tract_stations.empty:
            continue
        tract_combined = tract_stations.merge(station_to_jobs_all, on="station_id")
        tract_combined["total_time"] = tract_combined["microtransit_time"] + tract_combined["transit_time"]
        tract_combined = tract_combined.dropna(subset=["total_time"])
        if tract_combined.empty:
            continue
        best = tract_combined.loc[tract_combined.groupby("job_tract_id")["total_time"].idxmin()]
        results.append(best)

    best_route_c = pd.concat(results, ignore_index=True)
    filtered = best_route_c[best_route_c["total_time"] <= 45]
    jobs_c = filtered.merge(
        nyc_tract_jobs[["tract_id", "total_jobs"]].rename(columns={"tract_id": "job_tract_id"}),
        on="job_tract_id", how="left"
    )
    jobs_c["total_jobs"] = jobs_c["total_jobs"].fillna(0)
    reachable_c = jobs_c.groupby("GEOID")["total_jobs"].sum()

    catchment_sweep_results.append({
        "catchment_min": cutoff,
        "tracts_below_n_stations": tracts_below_n,
        "avg_stations_available": counts.mean(),
        "median_reachable_jobs": reachable_c.median(),
    })
    print(f"Catchment={cutoff} min: median={reachable_c.median():,.0f}, "
          f"avg stations available={counts.mean():.1f}, tracts <{N_STATIONS} stations={tracts_below_n}")

catchment_sweep_df = pd.DataFrame(catchment_sweep_results)
print("\n", catchment_sweep_df)

In [ ]:
# Check every catchment value from 1 to 15 minutes, minimum needed to guarantee 3+ stations for all tracts
min_check_results = []

for cutoff in range(1, 16):
    capped = microtransit_raw_all[microtransit_raw_all["microtransit_time"] <= cutoff]
    counts = capped.groupby("GEOID").size()

    # Account for tracts that might not appear at all in `capped` (0 stations available)
    all_tract_counts = pd.Series(0, index=low_access_geoids)
    all_tract_counts.update(counts)

    tracts_below_n = (all_tract_counts < N_STATIONS).sum()
    min_check_results.append({
        "catchment_min": cutoff,
        "tracts_below_n_stations": tracts_below_n,
        "min_stations_any_tract": all_tract_counts.min()
    })

min_check_df = pd.DataFrame(min_check_results)
print(min_check_df.to_string(index=False))

# Find the exact first minute where the count hits zero
exact_minimum = min_check_df[min_check_df["tracts_below_n_stations"] == 0]["catchment_min"].min()
print(f"\nExact minimum catchment duration guaranteeing ≥{N_STATIONS} stations for all target tracts: {exact_minimum} minutes")

## 7.2 Check Station Availability per Tract at Each Duration

Confirms the key finding: 10 minutes is the shortest catchment at which
every target tract has at least 3 candidate stations (i.e., can fully
satisfy the top-3 routing assumption).

In [ ]:
print(catchment_sweep_df[["catchment_min", "tracts_below_n_stations"]])

first_zero_shortfall = catchment_sweep_df[catchment_sweep_df["tracts_below_n_stations"] == 0]["catchment_min"].min()
print(f"\nShortest catchment with zero tracts falling short of {N_STATIONS} stations: {first_zero_shortfall} min")

## 7.3 Results Table & Justification for 10-Minute Catchment

Median reachable jobs is stable within ~1% across the full 3–20 minute range
once the top-N restriction is applied — the catchment duration's main role is
ensuring enough candidate stations exist, not determining the magnitude of
the accessibility gain. 10 minutes is the minimum duration satisfying that
requirement for all 222 target tracts.

In [ ]:
catchment_sweep_df.to_csv(f"{data_dir}/sensitivity_catchment_duration.csv", index=False)
print(catchment_sweep_df.to_string(index=False))
print(f"\n→ Selected catchment: 10 minutes (minimum duration with 0 tracts below {N_STATIONS}-station requirement)")

# 8. SENSITIVITY ANALYSIS: STATION COUNT (TOP-N)

## 8.1 Sweep N (1–20), Fixed 10-Minute Catchment

Tests whether the top-3 station restriction is justified, by holding the
catchment duration fixed at 10 minutes (established in Section 6) and
varying only how many nearest candidate stations each tract may use.

In [ ]:
micdafsmiclsdafdsfdsafsdaasdgasdgdsgsdgasdfsdfsdfsdfdfsgfsddfgbsdfgdfgn_values = [1, 2, 3, 4, 5, 7, 10, 15, 20]
n_sweep_results = []

microtransit_10min = microtransit_raw_all[microtransit_raw_all["microtransit_time"] <= CATCHMENT_MINUTES].copy()

for n in n_values:
    top_n = (
        microtransit_10min.sort_values("microtransit_time")
        .groupby("GEOID")
        .head(n)
        .copy()
    )

    results = []
    for geoid in low_access_geoids:
        tract_stations = top_n[top_n["GEOID"] == geoid]
        if tract_stations.empty:
            continue
        tract_combined = tract_stations.merge(station_to_jobs_all, on="station_id")
        tract_combined["total_time"] = tract_combined["microtransit_time"] + tract_combined["transit_time"]
        tract_combined = tract_combined.dropna(subset=["total_time"])
        if tract_combined.empty:
            continue
        best = tract_combined.loc[tract_combined.groupby("job_tract_id")["total_time"].idxmin()]
        results.append(best)

    best_route_n = pd.concat(results, ignore_index=True)
    filtered = best_route_n[best_route_n["total_time"] <= 45]
    jobs_n = filtered.merge(
        nyc_tract_jobs[["tract_id", "total_jobs"]].rename(columns={"tract_id": "job_tract_id"}),
        on="job_tract_id", how="left"
    )
    jobs_n["total_jobs"] = jobs_n["total_jobs"].fillna(0)
    reachable_n = jobs_n.groupby("GEOID")["total_jobs"].sum()

    n_sweep_results.append({
        "n_stations": n,
        "median_reachable_jobs": reachable_n.median(),
        "mean_reachable_jobs": reachable_n.mean(),
    })
    print(f"N={n}: median={reachable_n.median():,.0f}")

n_sweep_df = pd.DataFrame(n_sweep_results)
print("\n", n_sweep_df)

## 8.2 Compare Against High-Access Benchmark (Plausibility Ceiling)

Identifies the point at which allowing more station choice produces an
implausible result — median accessibility exceeding tracts that were already
well-served, which would not be a credible outcome of a first-mile fix alone.

In [ ]:
benchmark = gdf_clean[gdf_clean["quadrant"] == "High Need, High Access"]["reachable_jobs"].median()
print(f"High-Access benchmark (median): {benchmark:,.0f}\n")

n_sweep_df["exceeds_benchmark"] = n_sweep_df["median_reachable_jobs"] > benchmark
print(n_sweep_df)

first_exceed = n_sweep_df[n_sweep_df["exceeds_benchmark"]]["n_stations"].min()
print(f"\nSmallest N where median exceeds benchmark: {first_exceed}")

## 8.3 Results Table & Justification for N=3

Results stay below the High-Access benchmark for N≤5, and exceed it starting
at N=7. Within the plausible range (N=1–5), gains increase steadily without a
clear inflection point, so N=3 is selected as a middle value consistent with
realistic small-shuttle service design (typically 2–4 fixed hub connections),
retaining meaningful headroom below the benchmark ceiling.

In [ ]:
n_sweep_df.to_csv(f"{data_dir}/sensitivity_station_count.csv", index=False)
print(n_sweep_df.to_string(index=False))
print(f"\n→ Selected station count: N=3 (safely below benchmark, consistent with realistic shuttle design)")

# 9. SENSITIVITY ANALYSIS: DISPATCH/WAIT TIME

## 9.1 Add Uniform Wait Buffer to Microtransit Leg

The baseline model (Section 5) assumes instant vehicle availability, which is
unrealistic for any real on-demand service. This section tests how results
change as a uniform dispatch/wait buffer is added to the microtransit leg,
using the finalized baseline parameters (10-min catchment, top-3 stations).

In [ ]:
# Pre-merge each tract's top-3 station data with the station-to-jobs matrix once,
# so the buffer sweep below only needs to re-filter, not re-merge, at each buffer value
merged_per_tract = []

for geoid in low_access_geoids:
    tract_stations = top_n_stations[top_n_stations["GEOID"] == geoid]
    if tract_stations.empty:
        continue
    tract_combined = tract_stations.merge(station_to_jobs_all, on="station_id")
    tract_combined = tract_combined.dropna(subset=["transit_time"])
    if tract_combined.empty:
        continue
    merged_per_tract.append((geoid, tract_combined))

print(f"Pre-merged data ready for {len(merged_per_tract)} tracts")

## 9.2 Sweep Buffer Values (0–15 min)

In [ ]:
import gc

buffer_scenarios = [0, 3, 5, 7, 10, 15]
dispatch_sweep_results = []

baseline_lookup = gdf_clean.set_index("GEOID")["reachable_jobs"]

for buf in buffer_scenarios:
    print(f"Processing buffer = {buf} min...")
    results = []

    for geoid, tract_combined in merged_per_tract:
        tc = tract_combined.copy()
        tc["total_time"] = tc["microtransit_time"] + buf + tc["transit_time"]
        tc = tc[tc["total_time"] <= 45]
        if tc.empty:
            continue
        best = tc.loc[tc.groupby("job_tract_id")["total_time"].idxmin()]
        results.append(best)

    if results:
        best_route_buf = pd.concat(results, ignore_index=True)
        jobs_buf = best_route_buf.merge(
            nyc_tract_jobs[["tract_id", "total_jobs"]].rename(columns={"tract_id": "job_tract_id"}),
            on="job_tract_id", how="left"
        )
        jobs_buf["total_jobs"] = jobs_buf["total_jobs"].fillna(0)
        reachable_buf = jobs_buf.groupby("GEOID")["total_jobs"].sum().reset_index()
        reachable_buf.columns = ["GEOID", "reachable_jobs_buf"]
    else:
        reachable_buf = pd.DataFrame(columns=["GEOID", "reachable_jobs_buf"])

    merged_compare = pd.DataFrame({"GEOID": low_access_geoids})
    merged_compare["baseline"] = merged_compare["GEOID"].map(baseline_lookup)
    merged_compare = merged_compare.merge(reachable_buf, on="GEOID", how="left")
    merged_compare["reachable_jobs_buf"] = merged_compare["reachable_jobs_buf"].fillna(0)
    merged_compare["pct_change"] = (
        (merged_compare["reachable_jobs_buf"] - merged_compare["baseline"]) / merged_compare["baseline"] * 100
    )

    dispatch_sweep_results.append({
        "buffer_min": buf,
        "median_pct_change": merged_compare["pct_change"].median(),
        "pct_tracts_worse_off": (merged_compare["pct_change"] < 0).mean() * 100,
    })

    del results
    gc.collect()

dispatch_sweep_df = pd.DataFrame(dispatch_sweep_results)
print("\n", dispatch_sweep_df)

## 9.4 Share of Tracts Worse Off vs. Buffer

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style = "darkgrid", font = "DejaVu Serif")

fig, ax1 = plt.subplots(figsize=(12, 6))

# Left axis
sns.lineplot(
    data=dispatch_sweep_df,
    x="buffer_min",
    y="median_pct_change",
    marker="o",
    linewidth=1,
    zorder=1,
    ax=ax1,
    color="#c51E3a"
)

ax1.set_xlabel("Dispatch/Wait Buffer (minutes)")
ax1.set_ylabel("Median % Change in Accessibility")
ax1.tick_params(axis="y", labelcolor="#C51E3A")
ax1.axhline(0, color="gray", linestyle="--", linewidth=1)

# Right axis
ax2 = ax1.twinx()

sns.lineplot(
    data=dispatch_sweep_df,
    x="buffer_min",
    y="pct_tracts_worse_off",
    marker="s",
    linewidth=1,
    linestyle="--",
    ax=ax2,
    zorder=3,
    color="#26619c"
)

ax2.set_ylabel("% of Tracts Worse Off Than Baseline")
ax2.tick_params(axis="y", labelcolor="#26619C")

plt.title("Sensitivity of Microtransit Accessibility Gains to Dispatch Wait Time")

plt.show()

## 9.5 Break-Even Threshold

Identifies the buffer value at which median accessibility gain crosses zero —
the maximum dispatch time a real microtransit service could sustain while
still delivering a net positive benefit to the median target tract.

In [ ]:
# Find the two rows that bracket the zero-crossing
sorted_df = dispatch_sweep_df.sort_values("buffer_min").reset_index(drop=True)

crossing_buffer = None
for i in range(len(sorted_df) - 1):
    y1, y2 = sorted_df.loc[i, "median_pct_change"], sorted_df.loc[i + 1, "median_pct_change"]
    x1, x2 = sorted_df.loc[i, "buffer_min"], sorted_df.loc[i + 1, "buffer_min"]
    if y1 >= 0 and y2 < 0:
        # Linear interpolation to estimate where y crosses zero
        crossing_buffer = x1 + (0 - y1) / (y2 - y1) * (x2 - x1)
        break

print(f"Estimated break-even dispatch time: {crossing_buffer:.1f} minutes")
print(f"(Between {x1} min at {y1:.1f}% and {x2} min at {y2:.1f}%)")

# 10. SUMMARY & EXPORT

## 10.1 Consolidated Results Table (All Sensitivity Analyses)

In [ ]:
print("=" * 60)
print("BASELINE MICROTRANSIT MODEL (10-min catchment, N=3 stations, 0-min buffer)")
print("=" * 60)
print(reachable_jobs_microtransit["reachable_jobs_microtransit"].describe())
print(f"\nHigh-Access benchmark (median): {gdf_clean[gdf_clean['quadrant'] == 'High Need, High Access']['reachable_jobs'].median():,.0f}")
print(f"Pre-intervention baseline (median): {gdf_clean[gdf_clean['quadrant'] == 'High Need, Low Access']['reachable_jobs'].median():,.0f}")

print("\n" + "=" * 60)
print("SENSITIVITY: CATCHMENT DURATION")
print("=" * 60)
print(catchment_sweep_df.to_string(index=False))

print("\n" + "=" * 60)
print("SENSITIVITY: STATION COUNT (N)")
print("=" * 60)
print(n_sweep_df.to_string(index=False))

print("\n" + "=" * 60)
print("SENSITIVITY: DISPATCH/WAIT TIME")
print("=" * 60)
print(dispatch_sweep_df.to_string(index=False))

## 10.2 Final Figures

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(catchment_sweep_df["catchment_min"], catchment_sweep_df["median_reachable_jobs"], marker="o")
axes[0].set_title("Median Reachable Jobs vs. Catchment Duration")
axes[0].set_xlabel("Catchment (minutes)")
axes[0].set_ylabel("Median Reachable Jobs")

axes[1].plot(n_sweep_df["n_stations"], n_sweep_df["median_reachable_jobs"], marker="o", color="green")
axes[1].axhline(gdf_clean[gdf_clean["quadrant"] == "High Need, High Access"]["reachable_jobs"].median(),
                color="red", linestyle="--", label="High-Access benchmark")
axes[1].set_title("Median Reachable Jobs vs. N Stations")
axes[1].set_xlabel("N (top stations allowed)")
axes[1].legend()

axes[2].plot(dispatch_sweep_df["buffer_min"], dispatch_sweep_df["median_pct_change"], marker="o", color="purple")
axes[2].axhline(0, color="gray", linestyle="--")
axes[2].set_title("Median % Change vs. Dispatch Buffer")
axes[2].set_xlabel("Buffer (minutes)")

plt.tight_layout()
plt.savefig(f"{data_dir}/all_sensitivity_summary.jpeg", dpi=300, bbox_inches="tight")
plt.show()

## 10.3 Export Final Datasets to Drive

In [ ]:
# Core results
reachable_jobs_microtransit.to_parquet(f"{data_dir}/FINAL_reachable_jobs_microtransit.parquet")
best_route.to_parquet(f"{data_dir}/FINAL_best_route_microtransit.parquet")

# Sensitivity tables (already saved individually in each section, consolidated copy here)
catchment_sweep_df.to_csv(f"{data_dir}/FINAL_sensitivity_catchment.csv", index=False)
n_sweep_df.to_csv(f"{data_dir}/FINAL_sensitivity_station_count.csv", index=False)
dispatch_sweep_df.to_csv(f"{data_dir}/FINAL_sensitivity_dispatch.csv", index=False)

print("All final datasets exported to Drive:")
print("  - FINAL_reachable_jobs_microtransit.parquet")
print("  - FINAL_best_route_microtransit.parquet")
print("  - FINAL_sensitivity_catchment.csv")
print("  - FINAL_sensitivity_station_count.csv")
print("  - FINAL_sensitivity_dispatch.csv")

# Data Reloads

In [ ]:
import pandas as pd
import geopandas as gpd

data_dir = "/content/drive/MyDrive/Colab Notebooks/Project Dream/Data"

# --- Section 2: Core data ---
gdf_clean = gpd.read_parquet(f"{data_dir}/gdf_transit_equity_score.parquet")
nyc_tract_jobs = gpd.read_parquet(f"{data_dir}/nyc_tract_jobs.parquet")

# --- Section 3: Raw microtransit catchment matrix ---
microtransit_raw_all = pd.read_parquet(f"{data_dir}/microtransit_raw_uncapped_dedup.parquet")

# --- Section 4: Station-to-jobs matrix ---
station_to_jobs_all = pd.read_parquet(f"{data_dir}/station_to_jobs_all_dedup.parquet")

# --- Section 5: Baseline microtransit result ---
reachable_jobs_microtransit = pd.read_parquet(f"{data_dir}/reachable_jobs_microtransit_baseline.parquet")
best_route = pd.read_parquet(f"{data_dir}/best_route_baseline.parquet")

# --- Sensitivity sweep tables (Sections 6, 7, 8) ---
catchment_sweep_df = pd.read_csv(f"{data_dir}/sensitivity_catchment_duration.csv")
n_sweep_df = pd.read_csv(f"{data_dir}/sensitivity_station_count.csv")
dispatch_sweep_df = pd.read_csv(f"{data_dir}/sensitivity_dispatch_time.csv")

print("All saved data reloaded successfully! \n")
print(f"gdf_clean: {gdf_clean.shape}")
print(f"nyc_tract_jobs: {nyc_tract_jobs.shape}")
print(f"microtransit_raw_all: {microtransit_raw_all.shape}")
print(f"station_to_jobs_all: {station_to_jobs_all.shape}")
print(f"reachable_jobs_microtransit: {reachable_jobs_microtransit.shape}")
print(f"best_route: {best_route.shape}")
print(f"catchment_sweep_df: {catchment_sweep_df.shape}")
print(f"n_sweep_df: {n_sweep_df.shape}")
print(f"dispatch_sweep_df: {dispatch_sweep_df.shape}")

In [ ]:
# --- Derived variables needed for Sections 5.1–8 (cheap to rebuild, not separately saved) ---

low_access_geoids = gdf_clean[gdf_clean["quadrant"] == "High Need, Low Access"]["GEOID"]

CATCHMENT_MINUTES = 10
N_STATIONS = 3

microtransit_capped = microtransit_raw_all[
    microtransit_raw_all["microtransit_time"] <= CATCHMENT_MINUTES
].copy()

top_n_stations = (
    microtransit_capped.sort_values("microtransit_time")
    .groupby("GEOID")
    .head(N_STATIONS)
    .copy()
)

print(f"✓ Derived variables rebuilt")
print(f"low_access_geoids: {len(low_access_geoids)} tracts")
print(f"top_n_stations: {top_n_stations.shape}, max per tract: {top_n_stations.groupby('GEOID').size().max()}")